In [ ]:
import pandas as pd
import os
import re

In [ ]:
# !aws s3 cp s3://eko-ekoka-ai-project/data/generation_output/Feb29_generation_1 data/Feb29_generation_1 --sse='AES256' --recursive

In [ ]:
import datasets
dataset = datasets.Dataset.load_from_disk('data/pt_dataset_redact')

In [ ]:
df_ft = pd.DataFrame(dataset)

In [ ]:
df_ft

In [ ]:
df_ft_2 = df_ft.drop(['note_text_redacted', 'note_type_desc', 'token_nums'], axis=1)

In [ ]:
df_ft_2

In [ ]:
pt_dataset_chat = datasets.Dataset.from_pandas(df_ft_2)

In [ ]:
pt_dataset_chat

In [ ]:
pt_dataset_chat = pt_dataset_chat.shuffle(seed=36)

In [ ]:
def collate_to_inputs(example):
    sys_prompt = f"""<<SYS>>\nYou are a helpful, respectful, and honest assistant in a pediatric rehabilitation clinic. You follow these rules:
1. Follow directions meticulously.
2. Write in point-form. Do not write in paragraphs.
3. Do not produce any extra text. Only write what is asked for.
4. Clearly distinguish between reporting, observations, and goals.
5. Answer as helpfully as possible, while being safe. <</SYS>>
"""
    instruction = "Here is a Progress Note in SOAP format."

    training_text = f"<s>[INST]{sys_prompt} {instruction}\n{example['input_text']}[/INST]"

    example['training_text'] = training_text
    return example

In [ ]:
pt_dataset_chat = pt_dataset_chat.map(collate_to_inputs)

In [ ]:
pt_dataset_chat.save_to_disk('data/pt_dataset_chat')

In [ ]:
pt_dataset_chat[135]

In [ ]:
! aws s3 cp data/pt_dataset_chat s3://eko-ekoka-ai-project/data/pt_dataset_chat --sse='AES256' --recursive